# Tutorial 4: LSTM and RNN

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import time
import copy

# Configure tqdm for pandas operations
tqdm.pandas()

print(f"✓ PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")


✓ PyTorch version: 2.6.0+cu124
✓ Device: cuda


In [2]:
# Load data efficiently
flat_filepath = r'dataset/data/FlatFiles/NGAsub_MegaFlatfile_RotD50_050_R211022_public.xlsx'

# Helper function to convert period column name to float
def period_to_float(col_name):
    """Convert column name like T0pt010S to 0.010"""
    period_str = col_name[1:-1].replace('pt', '.')
    return float(period_str)

# Get all column names first
print("Reading column names")
df_temp = pd.read_excel(flat_filepath, engine='calamine', nrows=0)
all_cols = df_temp.columns.tolist()

# Find period columns <= 4.0 seconds only
period_cols = [col for col in all_cols if col.startswith('T') and col.endswith('S')]
selected_period_cols = [col for col in period_cols if period_to_float(col) <= 4.0]
selected_period_cols_sorted = sorted(selected_period_cols, key=period_to_float)

# Define columns to load
selected_columns = [
    'Earthquake_Magnitude',
    'Rjb_km',
    'Vs30_Selected_for_Analysis_m_s',
    'Fault_Type',
    'PGA_g',
    'PGV_cm_sec'
] + selected_period_cols_sorted

# Define dtypes for faster loading (avoid type inference)
dtype_dict = {
    'Earthquake_Magnitude': 'float32',
    'Rjb_km': 'float32',
    'Vs30_Selected_for_Analysis_m_s': 'float32',
    'Fault_Type': 'int8',
    'PGA_g': 'float32',
    'PGV_cm_sec': 'float32'
}
# Add period columns as float32
for col in selected_period_cols_sorted:
    dtype_dict[col] = 'float32'

print(f"Loading {len(selected_columns)} columns (6 base features + {len(selected_period_cols_sorted)} periods)...")
print(f"Period range: {period_to_float(selected_period_cols_sorted[0]):.3f}s to {period_to_float(selected_period_cols_sorted[-1]):.3f}s")

# Load with dtype specifications for faster processing
df = pd.read_excel(flat_filepath, engine='calamine', usecols=selected_columns, dtype=dtype_dict)

print(f"✓ Loaded data shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Reading column names
Loading 96 columns (6 base features + 90 periods)...
Period range: 0.010s to 4.000s
✓ Loaded data shape: (71340, 96)
  Memory usage: 25.92 MB


In [3]:
# Data Cleaning: Remove outliers and invalid values
print("="*80)
print("DATA CLEANING")
print("="*80)
print(f"\nOriginal shape: {df.shape}")

# Filter 1: Magnitude >= 4
df = df[df['Earthquake_Magnitude'] >= 4]
print(f"After M >= 4 filter: {df.shape}")

# Filter 2: Distance 0-500 km
df = df[(df['Rjb_km'] > 0) & (df['Rjb_km'] <= 500)]
print(f"After distance filter: {df.shape}")

# Filter 3: Remove rows with Vs30 <= 0
df = df[df['Vs30_Selected_for_Analysis_m_s'] > 0]
print(f"After Vs30 > 0 filter: {df.shape}")

# Filter 4: PGA realistic range (0-5g)
df = df[(df['PGA_g'] > 0) & (df['PGA_g'] < 5)]
print(f"After PGA filter: {df.shape}")

# Filter 5: PGV realistic range (0-500 cm/s)
df = df[(df['PGV_cm_sec'] > 0) & (df['PGV_cm_sec'] < 500)]
print(f"After PGV filter: {df.shape}")

# Filter 6: Remove rows with invalid spectral values (with progress bar)
print(f"\nFiltering {len(selected_period_cols_sorted)} spectral acceleration columns...")
for col in tqdm(selected_period_cols_sorted, desc="Filtering Sa columns"):
    df = df[(df[col] > 0) & (df[col] < 5)]

print(f"After spectral value filters: {df.shape}")

# Reset index
df = df.reset_index(drop=True)
print(f"\n✓ Final cleaned data: {len(df):,} earthquakes")
print("="*80)

DATA CLEANING

Original shape: (71340, 96)
After M >= 4 filter: (68473, 96)
After distance filter: (55298, 96)
After Vs30 > 0 filter: (55059, 96)
After PGA filter: (54972, 96)
After PGV filter: (54972, 96)

Filtering 90 spectral acceleration columns...


Filtering Sa columns: 100%|██████████| 90/90 [00:00<00:00, 418.65it/s]

After spectral value filters: (54967, 96)

✓ Final cleaned data: 54,967 earthquakes


# Multi-Output ANN for Ground Motion Prediction

In [4]:
df['Rjb_km'].describe()

count    54967.000000
mean       200.558365
std        120.732170
min          0.005944
25%        103.719273
50%        181.756287
75%        283.649216
max        499.997467
Name: Rjb_km, dtype: float64

In [5]:
df['Vs30_Selected_for_Analysis_m_s'].describe()

count    54967.000000
mean       429.677521
std        216.375931
min         53.000000
25%        270.000000
50%        389.600006
75%        527.754242
max       2229.800049
Name: Vs30_Selected_for_Analysis_m_s, dtype: float64

In [6]:
# Data Processing: Create log-transformed features and targets
print("="*80)
print("DATA PREPROCESSING")
print("="*80)

df = df.reset_index(drop=True)

# Create log10-transformed features (will be used as inputs)
print("\nCreating log-transformed input features...")
df['log10_Rjb_km'] = np.log10(df['Rjb_km'])
df['log10_Vs30'] = np.log10(df['Vs30_Selected_for_Analysis_m_s'])
print("  ✓ log10_Rjb_km, ✓ log10_Vs30")

# Create log10-transformed targets (will be used as outputs)
df['log10_PGA_g'] = np.log10(df['PGA_g'])
df['log10_PGV_cm_sec'] = np.log10(df['PGV_cm_sec'])
print("  ✓ log10_PGA_g, ✓ log10_PGV_cm_sec")

# Add log10 for all spectral acceleration periods (with progress bar)
print(f"\nCreating log10 for {len(selected_period_cols_sorted)} spectral periods...")
for col in tqdm(selected_period_cols_sorted, desc="Log-transforming Sa"):
    df[f'log10_{col}'] = np.log10(df[col])

# Verify no inf/nan values after log transformation
print("\nVerifying data integrity...")
log_feature_cols = ['log10_Rjb_km', 'log10_Vs30']
log_target_cols = ['log10_PGA_g', 'log10_PGV_cm_sec'] + [f'log10_{col}' for col in selected_period_cols_sorted]

# Check for inf/nan in log-transformed columns
inf_nan_count = 0
print("Checking for inf/nan values in log-transformed columns...")
for col in tqdm(log_feature_cols + log_target_cols, desc="Validating columns"):
    invalid_mask = ~np.isfinite(df[col])
    if invalid_mask.any():
        inf_nan_count += invalid_mask.sum()
        df = df[~invalid_mask]

if inf_nan_count > 0:
    print(f"  ⚠ Removed {inf_nan_count} rows with inf/nan values")
    df = df.reset_index(drop=True)
else:
    print(f"  ✓ All values are finite and valid")

# Store column lists for later use
input_feature_cols = ['Earthquake_Magnitude', 'Rjb_km', 'log10_Rjb_km', 'log10_Vs30', 'Fault_Type']
output_target_cols = log_target_cols


DATA PREPROCESSING

Creating log-transformed input features...
  ✓ log10_Rjb_km, ✓ log10_Vs30
  ✓ log10_PGA_g, ✓ log10_PGV_cm_sec

Creating log10 for 90 spectral periods...


Log-transforming Sa: 100%|██████████| 90/90 [00:00<00:00, 348.20it/s]



Verifying data integrity...
Checking for inf/nan values in log-transformed columns...


Validating columns: 100%|██████████| 94/94 [00:00<00:00, 6002.99it/s]

  ✓ All values are finite and valid


In [7]:
# Train/test split and feature scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

# Prepare X and y from dataframe
X = df[input_feature_cols].values
y = df[output_target_cols].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target matrix shape: {y.shape}")

# Split data (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n✓ Training samples: {X_train.shape[0]:,}")
print(f"✓ Testing samples: {X_test.shape[0]:,}")
print(f"✓ Split ratio: 80% train / 20% test")

# Standardize features (critical for neural networks)
print("\nApplying feature standardization...")
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Also scale targets (helps with training stability)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print(f"✓ X_train_scaled: {X_train_scaled.shape}")
print(f"✓ y_train_scaled: {y_train_scaled.shape}")
print(f"✓ Scalers fitted and ready")

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE")
print("="*80)

TRAIN-TEST SPLIT

Feature matrix shape: (54967, 5)
Target matrix shape: (54967, 92)

✓ Training samples: 43,973
✓ Testing samples: 10,994
✓ Split ratio: 80% train / 20% test

Applying feature standardization...
✓ X_train_scaled: (43973, 5)
✓ y_train_scaled: (43973, 92)
✓ Scalers fitted and ready

DATA PREPARATION COMPLETE


In [8]:
class LSTM(nn.Module):
    """
    Hybrid model combining static predictions (PGA, PGV) with autoregressive LSTM for spectral accelerations.
    
    Architecture:
    - PGA/PGV: Direct prediction from base features
    - Sa values: Autoregressive LSTM that uses previous Sa predictions + base features
    """
    def __init__(self, input_size=5, hidden_size=64, num_sa_outputs=90):
        super(LSTM, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_sa_outputs = num_sa_outputs
        
        # PGA prediction head
        self.pga_head = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1)
        )
        
        # PGV prediction head
        self.pgv_head = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1)
        )
        
        # LSTM for sequential Sa prediction (processes previous Sa values)
        self.lstm = nn.LSTM(
            input_size=1,              # Previous Sa value
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        
        # Sa predictor: combines LSTM hidden state + base features
        self.sa_predictor = nn.Sequential(
            nn.Linear(hidden_size + input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, x, y_sa=None):
        """
        Forward pass with different features for different outputs.
        
        Args:
            x: (batch_size, 5) - Base features
            y_sa: (batch_size, 90) - Ground truth Sa values from data.
                  If provided: uses actual previous Sa as LSTM input (training, single call)
                  If None: uses predicted Sa fed back (inference, sequential loop)
        
        Returns:
            (batch_size, 92) - [PGA, PGV, Sa1, Sa2, ..., Sa90]
        """
        batch_size = x.size(0)
        dev = x.device
        
        # Predict PGA and PGV (non-sequential, base features only)
        pga = self.pga_head(x)  # (batch, 1)
        pgv = self.pgv_head(x)  # (batch, 1)
        
        if y_sa is not None:
            # TRAINING: Use actual previous Sa from data as LSTM input
            # Shift right: [0, Sa1, Sa2, ..., Sa89] → predicts [Sa1, Sa2, ..., Sa90]
            prev_sa_seq = torch.cat([
                torch.zeros(batch_size, 1, device=dev),
                y_sa[:, :-1]
            ], dim=1).unsqueeze(2)  # (batch, 90, 1)
            
            # Single LSTM call for all 90 steps
            lstm_out, _ = self.lstm(prev_sa_seq)  # (batch, 90, hidden)
            
            # Broadcast base features: (batch, 5) → (batch, 90, 5)
            x_expanded = x.unsqueeze(1).expand(-1, self.num_sa_outputs, -1)
            
            # Combine LSTM output + base features at every time step
            combined = torch.cat([lstm_out, x_expanded], dim=2)  # (batch, 90, hidden+5)
            
            # Predict all Sa at once
            sa_tensor = self.sa_predictor(combined).squeeze(-1)  # (batch, 90)
        else:
            # INFERENCE: No ground truth, use predictions fed back
            sa_predictions = []
            h = torch.zeros(2, batch_size, self.hidden_size, device=dev)
            c = torch.zeros(2, batch_size, self.hidden_size, device=dev)
            prev_sa = torch.zeros(batch_size, 1, 1, device=dev)
            
            for i in range(self.num_sa_outputs):
                lstm_out, (h, c) = self.lstm(prev_sa, (h, c))
                combined = torch.cat([lstm_out.squeeze(1), x], dim=1)
                sa_i = self.sa_predictor(combined).squeeze(-1)
                sa_predictions.append(sa_i)
                prev_sa = sa_i.unsqueeze(1).unsqueeze(2)
            
            sa_tensor = torch.stack(sa_predictions, dim=1)
        
        # Combine: [PGA, PGV, Sa1..Sa90]
        outputs = torch.cat([pga, pgv, sa_tensor], dim=1)  # (batch, 92)
        return outputs

print("✓ LSTM model defined")
print("  - Training: Previous Sa from data → single LSTM call (fast)")
print("  - Inference: Predicted Sa fed back → sequential loop")

✓ LSTM model defined
  - Training: Previous Sa from data → single LSTM call (fast)
  - Inference: Predicted Sa fed back → sequential loop


In [9]:
def train(X_train, y_train, X_val, y_val, 
          hidden_size=20, lr=0.001, batch_size=64, 
          num_epochs=50, patience=10, verbose=False):
    """
    Train LSTM model with given hyperparameters.
    
    Args:
        X_train, y_train: Training data (scaled)
        X_val, y_val: Validation data (scaled)
        hidden_size: LSTM hidden dimension
        lr: Learning rate
        batch_size: Mini-batch size
        epochs: Maximum training epochs
        patience: Early stopping patience
        verbose: Print training progress
        
    Returns:
        dict: {
            'model': trained model,
            'train_loss': final training loss,
            'val_loss': final validation loss,
            'best_epoch': epoch with best val loss,
            'train_history': loss per epoch,
            'val_history': val loss per epoch
        }
    """
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val).to(device)
    
    # Create DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Initialize model
    model = LSTM(input_size=X_train.shape[1], hidden_size=hidden_size, num_sa_outputs=y_train.shape[1]-2).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Training tracking
    train_history = []
    val_history = []
    best_model_state = None
    best_epoch = 0
    patience_counter = 0
    best_val_loss = float('inf')
    
    # Training loop
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            # Pass ground truth Sa (columns 2:) as LSTM input
            outputs = model(batch_X, y_sa=batch_y[:, 2:])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_history.append(train_loss)
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor, y_sa=y_val_tensor[:, 2:])
            val_loss = criterion(val_outputs, y_val_tensor).item()
        val_history.append(val_loss)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Print progress
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    return {
        'model': model,
        'train_loss': train_history[best_epoch],
        'val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'train_history': train_history,
        'val_history': val_history
    }

print("✓ Training function defined")

✓ Training function defined


In [10]:
# Install and import wandb
import wandb

# Split train into train/val for hyperparameter search
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train_scaled, y_train_scaled, test_size=0.2, random_state=42
)

print(f"✓ Train set: {X_train_part.shape[0]:,} samples")
print(f"✓ Validation set: {X_val.shape[0]:,} samples")
print(f"✓ Test set: {X_test_scaled.shape[0]:,} samples (held out for final evaluation)")


✓ Train set: 35,178 samples
✓ Validation set: 8,795 samples
✓ Test set: 10,994 samples (held out for final evaluation)


# Hyperparameter Search (Sequential)

We'll search for optimal hyperparameters sequentially:
1. **Step 1:** Find best batch_size (fix lr=0.002, hidden_size=10)
2. **Step 2:** Find best learning_rate (use best batch_size, fix hidden_size=10)
3. **Step 3:** Find best hidden_size (use best batch_size and learning_rate)

Total runs: 4 + 4 + 5 = **13 runs** (much faster than grid search)

In [11]:
# Initialize W&B for sequential search
wandb.init(project="tutorial4-lstm", name="sequential_hyperparam_search", reinit=True)

# Define search space
batch_sizes = [16, 32, 64, 128]
learning_rates = [0.002, 0.001, 0.0005, 0.0001]
hidden_sizes = [10, 15, 20, 25, 30]

# Training hyperparameters
num_epochs = 50
patience = 10

print("="*80)
print("SEQUENTIAL HYPERPARAMETER SEARCH")
print("="*80)
print(f"Search space:")
print(f"  Batch sizes: {batch_sizes}")
print(f"  Learning rates: {learning_rates}")
print(f"  Hidden sizes: {hidden_sizes}")
print(f"\nTraining config:")
print(f"  Max epochs: {num_epochs}")
print(f"  Early stopping patience: {patience}")
print(f"  Total runs: {len(batch_sizes) + len(learning_rates) + len(hidden_sizes)}")
print("="*80)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/prem/.netrc.


wandb: Currently logged in as: premsuggu11 (prem11suggu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


SEQUENTIAL HYPERPARAMETER SEARCH
Search space:
  Batch sizes: [16, 32, 64, 128]
  Learning rates: [0.002, 0.001, 0.0005, 0.0001]
  Hidden sizes: [10, 15, 20, 25, 30]

Training config:
  Max epochs: 50
  Early stopping patience: 10
  Total runs: 13


In [12]:
# STEP 1: Search for best batch size
print("\n" + "="*80)
print("STEP 1: SEARCHING BATCH SIZE")
print("="*80)
print(f"Fixed: lr={learning_rates[0]}, hidden_size={hidden_sizes[0]}")

bs_results = []
for bs in tqdm(batch_sizes, desc="Batch size search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=hidden_sizes[0],
        lr=learning_rates[0],
        batch_size=bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )
    
    bs_results.append({
        'batch_size': bs,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss'],
        'best_epoch': result['best_epoch']
    })
    
    # Log to W&B
    wandb.log({
        'step': 'batch_size_search',
        'batch_size': bs,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss']
    })
    
    print(f"  batch_size={bs:3d} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best batch size
best_bs = min(bs_results, key=lambda x: x['val_loss'])['batch_size']
best_bs_loss = min(bs_results, key=lambda x: x['val_loss'])['val_loss']

print(f"\n✓ Best batch_size: {best_bs} (val_loss={best_bs_loss:.6f})")
wandb.log({'best_batch_size': best_bs, 'best_bs_val_loss': best_bs_loss})


STEP 1: SEARCHING BATCH SIZE
Fixed: lr=0.002, hidden_size=10


Batch size search:  25%|██▌       | 1/4 [03:15<09:46, 195.34s/it]

  batch_size= 16 → val_loss=0.019989, epochs=2


Batch size search:  50%|█████     | 2/4 [05:29<05:19, 159.52s/it]

  batch_size= 32 → val_loss=0.018484, epochs=8


Batch size search:  75%|███████▌  | 3/4 [07:14<02:14, 134.49s/it]

  batch_size= 64 → val_loss=0.019653, epochs=17


Batch size search: 100%|██████████| 4/4 [08:06<00:00, 121.64s/it]

  batch_size=128 → val_loss=0.019984, epochs=14

✓ Best batch_size: 32 (val_loss=0.018484)


In [13]:
# STEP 2: Search for best learning rate
print("\n" + "="*80)
print("STEP 2: SEARCHING LEARNING RATE")
print("="*80)
print(f"Fixed: batch_size={best_bs}, hidden_size={hidden_sizes[0]}")

lr_results = []
for lr in tqdm(learning_rates, desc="Learning rate search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=hidden_sizes[0],
        lr=lr,
        batch_size=best_bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )
    
    lr_results.append({
        'learning_rate': lr,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss'],
        'best_epoch': result['best_epoch']
    })
    
    # Log to W&B
    wandb.log({
        'step': 'learning_rate_search',
        'learning_rate': lr,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss']
    })
    
    print(f"  lr={lr:.6f} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best learning rate
best_lr = min(lr_results, key=lambda x: x['val_loss'])['learning_rate']
best_lr_loss = min(lr_results, key=lambda x: x['val_loss'])['val_loss']

print(f"\n✓ Best learning_rate: {best_lr} (val_loss={best_lr_loss:.6f})")
wandb.log({'best_learning_rate': best_lr, 'best_lr_val_loss': best_lr_loss})


STEP 2: SEARCHING LEARNING RATE
Fixed: batch_size=32, hidden_size=10


Learning rate search:  25%|██▌       | 1/4 [02:21<07:05, 141.93s/it]

  lr=0.002000 → val_loss=0.019252, epochs=9


Learning rate search:  50%|█████     | 2/4 [04:24<04:20, 130.28s/it]

  lr=0.001000 → val_loss=0.020355, epochs=6


Learning rate search:  75%|███████▌  | 3/4 [1:18:32<35:01, 2101.89s/it]

  lr=0.000500 → val_loss=0.019079, epochs=16


Learning rate search: 100%|██████████| 4/4 [1:22:02<00:00, 1230.61s/it]

  lr=0.000100 → val_loss=0.020698, epochs=44

✓ Best learning_rate: 0.0005 (val_loss=0.019079)


In [14]:
# STEP 3: Search for best hidden size
print("\n" + "="*80)
print("STEP 3: SEARCHING HIDDEN SIZE")
print("="*80)
print(f"Fixed: batch_size={best_bs}, lr={best_lr}")

hs_results = []
for hs in tqdm(hidden_sizes, desc="Hidden size search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=hs,
        lr=best_lr,
        batch_size=best_bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )
    
    hs_results.append({
        'hidden_size': hs,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss'],
        'best_epoch': result['best_epoch']
    })
    
    # Log to W&B
    wandb.log({
        'step': 'hidden_size_search',
        'hidden_size': hs,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss']
    })
    
    print(f"  hidden_size={hs:2d} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best hidden size
best_hs = min(hs_results, key=lambda x: x['val_loss'])['hidden_size']
best_hs_loss = min(hs_results, key=lambda x: x['val_loss'])['val_loss']

print(f"\n✓ Best hidden_size: {best_hs} (val_loss={best_hs_loss:.6f})")
wandb.log({'best_hidden_size': best_hs, 'best_hs_val_loss': best_hs_loss})


STEP 3: SEARCHING HIDDEN SIZE
Fixed: batch_size=32, lr=0.0005


Hidden size search:  20%|██        | 1/5 [01:17<05:08, 77.20s/it]

  hidden_size=10 → val_loss=0.023192, epochs=6


Hidden size search:  40%|████      | 2/5 [03:16<05:05, 101.75s/it]

  hidden_size=15 → val_loss=0.018712, epochs=18


...

  hidden_size=20 → val_loss=0.015740, epochs=23


Hidden size search:  80%|████████  | 4/5 [09:16<02:38, 158.18s/it]

  hidden_size=25 → val_loss=0.013664, epochs=49


Hidden size search: 100%|██████████| 5/5 [12:48<00:00, 153.78s/it]

  hidden_size=30 → val_loss=0.013918, epochs=32

✓ Best hidden_size: 25 (val_loss=0.013664)


In [15]:
# Display final best configuration
print("\n" + "="*80)
print("BEST CONFIGURATION")
print("="*80)
print(f"Batch size:     {best_bs}")
print(f"Learning rate:  {best_lr}")
print(f"Hidden size:    {best_hs}")
print(f"Validation loss: {best_hs_loss:.6f}")
print("="*80)

# Log final config to W&B
wandb.summary['final_batch_size'] = best_bs
wandb.summary['final_learning_rate'] = best_lr
wandb.summary['final_hidden_size'] = best_hs
wandb.summary['final_val_loss'] = best_hs_loss

# Finish W&B run
wandb.finish()

print("\n✓ Hyperparameter search complete!")
print(f"✓ View results at: https://wandb.ai/prem11suggu/tutorial4-lstm")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.



BEST CONFIGURATION
Batch size:     32
Learning rate:  0.0005
Hidden size:    25
Validation loss: 0.013664


wandb: ERROR Control-C detected -- Run data was not synced


KeyboardInterrupt: 

In [ ]:
# Plot hyperparameter search results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Batch size vs Val Loss
axes[0].plot([r['batch_size'] for r in bs_results], 
             [r['val_loss'] for r in bs_results], 'o-', linewidth=2, markersize=8)
axes[0].axvline(best_bs, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_bs}')
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('Validation Loss', fontsize=12)
axes[0].set_title('Batch Size vs Validation Loss', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Learning Rate vs Val Loss
axes[1].plot([r['learning_rate'] for r in lr_results], 
             [r['val_loss'] for r in lr_results], 'o-', linewidth=2, markersize=8, color='orange')
axes[1].axvline(best_lr, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_lr}')
axes[1].set_xlabel('Learning Rate', fontsize=12)
axes[1].set_ylabel('Validation Loss', fontsize=12)
axes[1].set_title('Learning Rate vs Validation Loss', fontsize=14, fontweight='bold')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Plot 3: Hidden Size vs Val Loss
axes[2].plot([r['hidden_size'] for r in hs_results], 
             [r['val_loss'] for r in hs_results], 'o-', linewidth=2, markersize=8, color='green')
axes[2].axvline(best_hs, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_hs}')
axes[2].set_xlabel('Hidden Size', fontsize=12)
axes[2].set_ylabel('Validation Loss', fontsize=12)
axes[2].set_title('Hidden Size vs Validation Loss', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig('assignments/4/hyperparam_search.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Plots saved to: assignments/4/hyperparam_search.png")